In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("load_type", "", "Load Type")
dbutils.widgets.text("merge_keys", "", "Merge Keys (comma-separated)")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type")
merge_keys = [k.strip() for k in dbutils.widgets.get("merge_keys").split(",") if k.strip()]

# Normalize and validate load type (empty defaults to FULL for backward compatibility)
load_type = (load_type or "FULL").strip().upper()
if load_type not in ("FULL", "APPEND", "MERGE"):
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")
if load_type == "MERGE" and not merge_keys:
    raise ValueError("merge_keys must be provided when load_type = 'MERGE'")

print("=" * 60)
print("[STEP 1] Notebook parameters")
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  table_name        : {table_name}")
print(f"  load_type         : {load_type}")
print(f"  merge_keys        : {merge_keys if merge_keys else 'N/A'}")
print("=" * 60)

In [0]:
source_table = f"bharath_dev_adb_p1.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print("[STEP 2] Reading bronze data")
print(f"  Source table : {source_table}")
print(f"  Filter       : {filter_condition}")
print(f"  Equivalent query:\n    SELECT * FROM {source_table}\n    WHERE {filter_condition}")

bronze_df = spark.read.table(source_table)
bronze_df = bronze_df.filter(filter_condition)

print(f"  Records read from bronze: {bronze_df.count()}")
display(bronze_df)

In [0]:
from pyspark.sql.functions import current_timestamp

print("[STEP 3] Adding audit columns: insert_timestamp, update_timestamp")

updated_bronze_df = bronze_df.withColumn("insert_timestamp", current_timestamp()) \
    .withColumn("update_timestamp", current_timestamp())

print(f"  Columns after transformation: {updated_bronze_df.columns}")
display(updated_bronze_df)

In [0]:
from delta.tables import DeltaTable

target_table = f"bharath_dev_adb_p1.silver.{table_name}"
column_list = ", ".join([f"`{c}`" for c in updated_bronze_df.columns])

print("[STEP 4] Writing to silver")
print(f"  Target table : {target_table}")
print(f"  Load type    : {load_type}")

if load_type == "FULL":
    # Full load: replace the target table contents
    print("  Action: Overwriting target table with incoming data")
    print(f"  Equivalent query:\n    INSERT OVERWRITE {target_table}\n    SELECT {column_list} FROM <source_df>")
    updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type == "APPEND":
    # Append: add incoming records to the target table
    print("  Action: Appending incoming data to target table")
    print(f"  Equivalent query:\n    INSERT INTO {target_table}\n    SELECT {column_list} FROM <source_df>")
    updated_bronze_df.write.mode("append").saveAsTable(target_table)

elif load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        # First run: create the target table from the incoming data
        print("  Target table does not exist - creating it from incoming data (initial load)")
        print(f"  Equivalent query:\n    CREATE TABLE {target_table} AS\n    SELECT {column_list} FROM <source_df>")
        updated_bronze_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        # Deduplicate source on merge keys to avoid multiple-match merge errors
        print("  Target table exists - performing MERGE (upsert)")
        source_df = updated_bronze_df.dropDuplicates(merge_keys)
        print(f"  Source records after dedup on {merge_keys}: {source_df.count()}")

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in merge_keys])
        print(f"  Merge condition: {merge_condition}")

        # On update, keep the original insert_timestamp and refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        # Build the equivalent MERGE SQL for visibility
        update_clause = ",\n        ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        merge_sql = (
            f"MERGE INTO {target_table} AS t\n"
            f"USING <source_df> AS s\n"
            f"ON {merge_condition}\n"
            f"WHEN MATCHED THEN UPDATE SET\n        {update_clause}\n"
            f"WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
            f"  VALUES ({insert_vals})"
        )
        print(f"  Equivalent query:\n{merge_sql}")

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())

        # Show merge metrics from the latest Delta history entry
        metrics = DeltaTable.forName(spark, target_table).history(1).select("operationMetrics").collect()[0][0]
        print("  Merge metrics:")
        print(f"    Rows inserted : {metrics.get('numTargetRowsInserted')}")
        print(f"    Rows updated  : {metrics.get('numTargetRowsUpdated')}")
        print(f"    Rows deleted  : {metrics.get('numTargetRowsDeleted')}")

print(f"  Target row count after load: {spark.table(target_table).count()}")
print(f"Load type '{load_type}' completed for {target_table}")

In [0]:
record_count = updated_bronze_df.count()
print(f"[STEP 5] Exiting notebook with processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))